# Study 3: Feature Set Comparison and SPQ Contribution

1. **Feature set comparison**: Train XGBoost on six feature sets (demographics, AQ only, EQ+SQ, SPQ only, all no AQ, all features); report AUROC/F1 per cohort.
2. **SPQ contribution**: Compare AUROC with vs without SPQ in C4 and CARD (DeLong or bootstrap).
3. **SHAP** (optional): Feature importance via SHAP when available.

In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd

_cwd = os.path.abspath(os.getcwd())
REPO_ROOT = os.path.dirname(_cwd) if os.path.basename(_cwd) == 'notebooks' else _cwd
if not os.path.isdir(os.path.join(REPO_ROOT, 'data')):
    REPO_ROOT = os.path.dirname(REPO_ROOT)
sys.path.insert(0, os.path.join(REPO_ROOT, 'src'))

from study_utils import (
    load_cohort_c4,
    load_cohort_card,
    load_cohort_ybt,
    train_with_cv,
    evaluate_model,
    bootstrap_ci_auroc,
    get_models,
    FEATURE_NAMES_45,
    FEATURE_NAMES_35,
    SPQ_COLS,
    DEMOGRAPHICS_FEATURES,
    AQ_ITEM_FEATURES,
    EQ_SQ_ONLY_FEATURES,
    SPQ_ITEM_FEATURES,
    RESULTS_DIR,
    RANDOM_STATE,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, f1_score

STUDY3_DIR = os.path.join(RESULTS_DIR, 'study3_features')
os.makedirs(os.path.join(STUDY3_DIR, 'shap_values'), exist_ok=True)

C4_PATH = os.path.join(REPO_ROOT, 'data', 'processed', 'data_c4_final_recreated_cleaned.csv')
CARD_ALIGNED_PATH = os.path.join(REPO_ROOT, 'data', 'processed', 'card_aligned.csv')
YBT_ALIGNED_PATH = os.path.join(REPO_ROOT, 'data', 'processed', 'ybt_aligned.csv')
YBT_RAW_PATH = os.environ.get('YBT_PATH', os.path.expanduser('~/Library/CloudStorage/OneDrive-UniversityofCambridge/Documents/PhD/data/YBT.csv'))

## 1. Load cohorts with AQ for feature-set experiments

In [ ]:
df_c4, feat_c4, target_c4 = load_cohort_c4(C4_PATH, age_min=18, age_max=55, balance_50_50=True, apply_aq_filter=True, keep_all_columns=True)
df_card = None
if os.path.isfile(CARD_ALIGNED_PATH):
    df_card, feat_card, target_card = load_cohort_card(CARD_ALIGNED_PATH, age_min=18, age_max=55, balance_50_50=True, apply_aq_filter=True)
else:
    feat_card, target_card = None, None
ybt_path = YBT_ALIGNED_PATH if os.path.isfile(YBT_ALIGNED_PATH) else YBT_RAW_PATH
df_ybt, feat_ybt, target_ybt = load_cohort_ybt(ybt_path, age_min=18, age_max=55, balance_50_50=True, apply_aq_filter=True)

def get_feature_set_columns(df, set_name, has_spq, has_aq=False):
    available = set(df.columns)
    if set_name == 'demographics':
        return [f for f in DEMOGRAPHICS_FEATURES if f in available]
    if set_name == 'aq_only' and has_aq:
        return [f for f in AQ_ITEM_FEATURES if f in available]
    if set_name == 'eq_sq_only':
        return [f for f in EQ_SQ_ONLY_FEATURES if f in available]
    if set_name == 'spq_only' and has_spq:
        return [f for f in SPQ_ITEM_FEATURES if f in available]
    if set_name == 'all_no_aq':
        base = FEATURE_NAMES_45 if has_spq else FEATURE_NAMES_35
        return [f for f in base if f in available and f not in AQ_ITEM_FEATURES and f != 'aq_total']
    if set_name == 'all_features':
        return [f for f in (FEATURE_NAMES_45 if has_spq else FEATURE_NAMES_35) if f in available]
    return []

## 2. Train and evaluate each feature set (5-fold CV AUROC)

In [ ]:
def cv_auroc_f1(X, y, n_splits=5):
    if X.shape[1] == 0 or len(np.unique(y)) < 2: return np.nan, np.nan
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X)
    model = get_models()['xgboost']
    probas = np.zeros_like(y, dtype=float)
    for train_idx, val_idx in skf.split(Xs, y):
        m = get_models()['xgboost']
        m.fit(Xs[train_idx], y[train_idx])
        probas[val_idx] = m.predict_proba(Xs[val_idx])[:, 1]
    auroc = roc_auc_score(y, probas)
    pred = (probas >= 0.5).astype(int)
    f1 = f1_score(y, pred, zero_division=0)
    return auroc, f1

feature_set_names = ['demographics', 'aq_only', 'eq_sq_only', 'spq_only', 'all_no_aq', 'all_features']
rows = []
cohorts_s3 = [('C4', df_c4, True, True, target_c4), ('Dataset3', df_ybt, False, True, target_ybt)]
if df_card is not None:
    cohorts_s3.insert(1, ('CARD', df_card, True, True, target_card))
for cohort_name, df, has_spq, has_aq, target_col in cohorts_s3:
    if df is None: continue
    for set_name in feature_set_names:
        cols = get_feature_set_columns(df, set_name, has_spq, has_aq)
        if not cols: continue
        X = df[cols].fillna(0).values
        y = df[target_col].values
        auroc, f1 = cv_auroc_f1(X, y)
        rows.append({'Cohort': cohort_name, 'Feature_Set': set_name, 'N_Features': len(cols), 'AUROC': auroc, 'F1': f1})

if rows:
    feat_comp = pd.DataFrame(rows)
    feat_comp.to_csv(os.path.join(STUDY3_DIR, 'feature_comparison_table.csv'), index=False)
    print(feat_comp.to_string(index=False))

## 3. SPQ contribution: with vs without SPQ (C4 and CARD)

In [ ]:
def spq_contribution(df, feature_names_full, target_col, cohort_name):
    feat_no_spq = [f for f in feature_names_full if f not in SPQ_COLS]
    X_full = df[feature_names_full].fillna(0).values
    X_no_spq = df[feat_no_spq].fillna(0).values
    y = df[target_col].values
    auroc_full, _ = cv_auroc_f1(X_full, y)
    auroc_no_spq, _ = cv_auroc_f1(X_no_spq, y)
    delta = auroc_full - auroc_no_spq
    return {'cohort': cohort_name, 'auroc_with_spq': auroc_full, 'auroc_without_spq': auroc_no_spq, 'delta_auroc': delta}

spq_results = {}
if df_c4 is not None:
    spq_results['c4'] = spq_contribution(df_c4, feat_c4, target_c4, 'c4')
if df_card is not None:
    spq_results['card'] = spq_contribution(df_card, feat_card, target_card, 'card')
with open(os.path.join(STUDY3_DIR, 'spq_contribution_analysis.json'), 'w') as f:
    json.dump(spq_results, f, indent=2)
print(json.dumps(spq_results, indent=2))

## 4. SHAP summary (optional)

Requires `pip install shap`. Run once per cohort with full features.

In [ ]:
try:
    import shap
    HAS_SHAP = True
except ImportError:
    HAS_SHAP = False
    print('SHAP not installed; skip SHAP cells or: pip install shap')

if HAS_SHAP and df_c4 is not None:
    X_c4 = df_c4[feat_c4].fillna(0)
    y_c4 = df_c4[target_c4].values
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X_c4)
    model = get_models()['xgboost']
    model.fit(Xs, y_c4)
    explainer = shap.TreeExplainer(model, Xs[:500])
    shap_vals = explainer.shap_values(Xs[:500])
    shap.summary_plot(shap_vals, X_c4.iloc[:500], show=False)
    import matplotlib.pyplot as _plt
    _plt.gcf().savefig(os.path.join(STUDY3_DIR, 'shap_values', 'c4_shap_summary.png'), dpi=150, bbox_inches='tight')
    _plt.close()
    print('Saved c4_shap_summary.png')